In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
from PIL import Image
import os

directory = "E:\IIT Year 2\Sem 1\DSGP\Images"

def convert_images_to_jpg(directory):
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        if os.path.isdir(folder_path):
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                try:
                    with Image.open(file_path) as img:
                        if img.format != 'JPEG':
                            new_file_path = file_path.rsplit('.', 1)[0] + '.jpg'
                            img.convert('RGB').save(new_file_path, 'JPEG')
                            os.remove(file_path)  
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")

convert_images_to_jpg(directory)


<>:4: SyntaxWarning: invalid escape sequence '\I'
<>:4: SyntaxWarning: invalid escape sequence '\I'
C:\Users\dulin\AppData\Local\Temp\ipykernel_25216\2396381816.py:4: SyntaxWarning: invalid escape sequence '\I'
  directory = "E:\IIT Year 2\Sem 1\DSGP\Images"


In [3]:
data_path = "E:\IIT Year 2\Sem 1\DSGP\Images"

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\dulin\AppData\Local\Temp\ipykernel_25216\39687889.py:1: SyntaxWarning: invalid escape sequence '\I'
  data_path = "E:\IIT Year 2\Sem 1\DSGP\Images"


In [4]:
dataset = tf.keras.utils.image_dataset_from_directory(
    data_path,
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(128, 128),
    shuffle=True
)


Found 3963 files belonging to 2 classes.


In [7]:
def split_dataset(dataset, train = 0.7, val = 0.2):
    dataset_size = len(dataset)
    training_size = int(dataset_size*train)
    validation_size = int(dataset_size*val)

    train_dataset = dataset.take(training_size)
    validation_dataset = dataset.skip(training_size).take(validation_size)
    test_dataset = dataset.skip(training_size+validation_size)

    return train_dataset, validation_dataset, test_dataset

In [8]:
train_dataset, validation_dataset, test_dataset = split_dataset(dataset)

In [9]:
for image_batch, label_batch in train_dataset.take(1):
    print(image_batch.shape)
    print(label_batch.shape)

(32, 128, 128, 3)
(32,)


In [16]:
from tensorflow.keras.applications import VGG19, ResNet50
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg19_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

In [17]:
train_dataset_vgg = train_dataset.map(lambda x, y: (vgg19_preprocess(x), y))
validation_dataset_vgg = validation_dataset.map(lambda x, y: (vgg19_preprocess(x), y))
test_dataset_vgg = test_dataset.map(lambda x, y: (vgg19_preprocess(x), y))

train_dataset_resnet = train_dataset.map(lambda x, y: (resnet_preprocess(x), y))
validation_dataset_resnet = validation_dataset.map(lambda x, y: (resnet_preprocess(x), y))
test_dataset_resnet = test_dataset.map(lambda x, y: (resnet_preprocess(x), y))

In [ ]:
def build_and_train_model(base_model, train_data, val_data, model_name):
    base_model.trainable = False  
    model = tf.keras.models.Sequential([
        base_model,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid') 
    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=['accuracy'])

    history = model.fit(train_data, epochs=10, validation_data=val_data)

    model.save(f"{model_name}.h5")  
    return model, history

In [19]:
pretrained_vgg19 = tf.keras.applications.VGG19(
    include_top=False,
    input_shape=(128, 128, 3),
    weights='imagenet',
    pooling='max'
)
vgg19_model, vgg19_history = build_and_train_model(pretrained_vgg19, train_dataset_vgg, validation_dataset_vgg, "vgg19_balcony_model")

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 291s 4us/step
Epoch 1/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 92s 1s/step - accuracy: 0.6918 - loss: 8.3391 - val_accuracy: 0.8724 - val_loss: 0.6338
Epoch 2/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.8610 - loss: 0.7155 - val_accuracy: 0.9089 - val_loss: 0.2368
Epoch 3/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.8748 - loss: 0.3338 - val_accuracy: 0.9193 - val_loss: 0.2081
Epoch 4/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.9045 - loss: 0.2581 - val_accuracy: 0.9414 - val_loss: 0.1820
Epoch 5/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 97s 1s/step - accuracy: 0.9071 - loss: 0.2073 - val_accuracy: 0.9427 - val_loss: 0.1467
Epoch 6/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.9375 - loss: 0.1435 - val_accuracy: 0.9622 - val_loss: 0.1119
Epoch 7/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.9500 - loss: 0.1481 - val_accuracy: 0.9609 - val_loss: 0.1125
Epoch 8/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 96s 1s/step - accuracy: 0.9

In [20]:
pretrained_resnet50 = tf.keras.applications.ResNet50(
    include_top=False,
    input_shape=(128, 128, 3),
    weights='imagenet',
    pooling='max'
)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 456s 5us/step


In [21]:
resnet50_model, resnet50_history = build_and_train_model(pretrained_resnet50, train_dataset_resnet, validation_dataset_resnet, "resnet50_balcony_model")

Epoch 1/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 52s 536ms/step - accuracy: 0.7504 - loss: 1.8355 - val_accuracy: 0.8945 - val_loss: 0.2448
Epoch 2/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 44s 509ms/step - accuracy: 0.8860 - loss: 0.2647 - val_accuracy: 0.9232 - val_loss: 0.2047
Epoch 3/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 44s 506ms/step - accuracy: 0.9182 - loss: 0.2046 - val_accuracy: 0.9349 - val_loss: 0.1728
Epoch 4/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 44s 510ms/step - accuracy: 0.9348 - loss: 0.1541 - val_accuracy: 0.9479 - val_loss: 0.1477
Epoch 5/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 45s 516ms/step - accuracy: 0.9356 - loss: 0.1584 - val_accuracy: 0.9414 - val_loss: 0.1652
Epoch 6/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 45s 513ms/step - accuracy: 0.9540 - loss: 0.1233 - val_accuracy: 0.9505 - val_loss: 0.1761
Epoch 7/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 45s 520ms/step - accuracy: 0.9563 - loss: 0.1056 - val_accuracy: 0.9609 - val_loss: 0.1175
Epoch 8/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 44s 510ms/step - accuracy: 0.9724 - loss: 0.0658 - val_accu

In [22]:
def evaluate_model(model, test_data, model_name):
    test_loss, test_accuracy = model.evaluate(test_data, verbose=1)
    print(f"\n{model_name} - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


In [23]:
evaluate_model(vgg19_model, test_dataset_vgg, "VGG19")

14/14 ━━━━━━━━━━━━━━━━━━━━ 17s 856ms/step - accuracy: 0.9394 - loss: 0.1596

VGG19 - Test Loss: 0.1981, Test Accuracy: 0.9345


In [24]:
evaluate_model(resnet50_model, test_dataset_resnet, "ResNet50")

14/14 ━━━━━━━━━━━━━━━━━━━━ 10s 377ms/step - accuracy: 0.9748 - loss: 0.0987

ResNet50 - Test Loss: 0.1094, Test Accuracy: 0.9684
